This code is used to compute the total time of different activites by participant in the PAAWS R1
dataset.

If you have any questions, please [email Veronika Potter](emailto:potter.v@northeastern.edu).

NOTE: this code was originally run in June 2025 using an unknown Python verison. The current
imlpementation yields slightly different numbers, we suspect due to rounding. The sheet
Table_4_Data.xlsx has both the old and new numbers in it.

In [ ]:
import pandas as pd
import numpy as np
import os

In [ ]:
FOLDER = "table_4_output" # TODO: change this path
LABEL_SETS = ["PA_Type", "Posture", "Contextual_Parameters", "High_Level_Behavior"]

# The participant IDs in the PAAWS R1 dataset
DATASETS = [10, 36, 37, 38, 39, 42, 44, 48, 49, 51, 58, 59, 87, 138, 139, 140, 235, 239, 240, 246]

**NOTE**: the following code was designed to be run on our local copy of the PAAWS dataset. You
may need to change path variables as neccesary to run the code on your local copy. Our local copy
stores the diffent labelsets seperately (e.g., only HLBs are stored in one file.)

In [ ]:
# Merges all labels (e.g., PA, Post, CP, and HLB) into a single dataframe.
ann_raw = {}
merged_dfs = {}

for ds in DATASETS:
    print(f"** Fetching data from DS_{ds}")
    path = f"./freelivingannotation/DS_{ds}/"
    for r, _, f in os.walk(path):
        for name in f:
            for label in LABEL_SETS:
                if (
                    label in name
                    and ".csv" in name
                    and "_corr" in name
                    and "combined" in name
                ):
                    csv = os.path.join(r, name)
                    if os.path.exists(csv):
                        ann_raw[ds, label] = pd.read_csv(
                            csv,
                            parse_dates=["START_TIME", "STOP_TIME"],
                            usecols=["PREDICTION", "START_TIME", "STOP_TIME"],
                        )

                        ann_raw[ds, label].rename(columns={"PREDICTION": label}, inplace=True)

                        if ds in merged_dfs.keys():
                            merged_dfs[ds] = pd.merge(
                                merged_dfs[ds],
                                ann_raw[ds, label],
                                how="outer",
                                on=["START_TIME", "STOP_TIME"],
                            )
                        else:
                            merged_dfs[ds] = ann_raw[ds, label]

print("\n***** All datasets are merged.")

** Fetching data from DS_10
** Fetching data from DS_36
** Fetching data from DS_37
** Fetching data from DS_38
** Fetching data from DS_39
** Fetching data from DS_42
** Fetching data from DS_44
** Fetching data from DS_48
** Fetching data from DS_49
** Fetching data from DS_51
** Fetching data from DS_58


/var/folders/_s/mj02y9ys3rnf1tp6lgr31syc0000gn/T/ipykernel_5428/4091610096.py:14: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ann_raw[ds, l] = pd.read_csv(
/var/folders/_s/mj02y9ys3rnf1tp6lgr31syc0000gn/T/ipykernel_5428/4091610096.py:14: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ann_raw[ds, l] = pd.read_csv(
/var/folders/_s/mj02y9ys3rnf1tp6lgr31syc0000gn/T/ipykernel_5428/4091610096.py:14: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ann_raw[ds, l] = pd.read_csv(
/var/folders/_s/mj02y9ys3rnf1tp6lgr31syc0000gn/T/ipykernel_5428/4091610096.py:14: UserWarning: Could not i

** Fetching data from DS_59
** Fetching data from DS_87
** Fetching data from DS_138
** Fetching data from DS_139
** Fetching data from DS_235
** Fetching data from DS_239
** Fetching data from DS_240
** Fetching data from DS_246
** Fetching data from DS_140

***** All datasets are merged.


In [ ]:
# Combines all lables from each category into a single string (PA*POS*CP*HLB). Adds these labels
# and their durations to each participants dataframes.
labels = []

for ds in merged_dfs.keys():
    print("** Combing labels from", ds)
    for i in range(merged_dfs[ds].shape[0]):
        temp_label = []
        for label in LABEL_SETS:
            pred = merged_dfs[ds].iloc[i][label]

            # handle nan data which should be marked unlabeled
            if str(pred) == "nan":
                pred = f"{label}_Unlabeled"

            if "High_Level_Behavior" in label or "Contextual" in label:
                temp = pred.split("|")
                temp.sort()
                if temp:
                    pred_sorted = "|".join(temp)
                    if pred_sorted not in labels:
                        temp_label.append(pred_sorted)
            else:
                temp_label.append(pred)

        lab = "*".join(temp_label)

        # Remove all labels that have either an unlabeled Posture or are about sensor syncing.
        if "PA_Type" not in lab and "Sensor" not in lab:
            if lab not in labels:
                labels.append(lab)

            # Add combined label and duration to a new col in the df.
            merged_dfs[ds].loc[i, "Merged"] = lab
            dur = (
                merged_dfs[ds].iloc[i]["STOP_TIME"]
                - merged_dfs[ds].iloc[i]["START_TIME"]
            ).total_seconds()
            merged_dfs[ds].loc[i, "Duration"] = dur

print("\n Unique activities: ", len(labels))

** Combing labels from 10
** Combing labels from 36
** Combing labels from 37
** Combing labels from 38
** Combing labels from 39
** Combing labels from 42
** Combing labels from 44
** Combing labels from 48
** Combing labels from 49
** Combing labels from 51
** Combing labels from 58
** Combing labels from 59
** Combing labels from 87
** Combing labels from 138
** Combing labels from 139
** Combing labels from 235
** Combing labels from 239
** Combing labels from 240
** Combing labels from 246
** Combing labels from 140
Unique activities:  5241


In [ ]:
# Save the output files with combined labels and total duration.
# NOTE: This cell is optional.

for key in merged_dfs.keys():
    merged_dfs[key].to_csv(f"./{FOLDER}/Merged_DS_{key}.csv")

In [ ]:
# Calculate total durations of each activity across participants. Saves a csv of the computed totals.

df = pd.DataFrame(columns=labels, index=merged_dfs.keys())
df.index.name = "Dataset"

for lab in df.columns:
    df[lab] = 0

# Helper variables to count the number of activities and lengths of bouts
bout_durs = []

for ds in merged_dfs.keys():
    print(f"** Computing durations from DS_{ds}")
    for i in range(merged_dfs[ds].shape[0]):
        curr_lab = merged_dfs[ds].loc[i, "Merged"]
        if str(curr_lab) != "nan":
            dur = merged_dfs[ds].loc[i, "Duration"]
            df.loc[ds, curr_lab] += int(dur)

            bout_durs.append(dur)

# df.to_csv(f"./{FOLDER}/PAAWS_R1_TOTAL_TIME.csv")

# Print stats about bout length.
print("\n** Bout fast facts **")
print("Number of bouts:", len(bout_durs))
print("Mean length of a bout (s):", np.average(bout_durs))
print("Median length of a bout (s):", np.median(bout_durs))
print("Min length of a bout (s):", np.amin(bout_durs))
print("Max length of a bout (s):", np.amax(bout_durs))
print("25th percentile length of a bout (s):", np.percentile(bout_durs, 25))
print("75th percentile length of a bout (s):", np.percentile(bout_durs, 75))

** Computing durations from DS_10
** Computing durations from DS_36
** Computing durations from DS_37
** Computing durations from DS_38
** Computing durations from DS_39
** Computing durations from DS_42
** Computing durations from DS_44
** Computing durations from DS_48
** Computing durations from DS_49
** Computing durations from DS_51
** Computing durations from DS_58
** Computing durations from DS_59
** Computing durations from DS_87
** Computing durations from DS_138
** Computing durations from DS_139
** Computing durations from DS_235
** Computing durations from DS_239
** Computing durations from DS_240
** Computing durations from DS_246
** Computing durations from DS_140

** Bout fast facts **
Number of bouts: 45130
Mean length of a bout (s): 124.72882029692
Median length of a bout (s): 33.0
Min length of a bout (s): 1.0
Max length of a bout (s): 12756.725
25th percentile length of a bout (s): 15.0
75th percentile length of a bout (s): 89.71575
